# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step template for loading and exploring the FAIR^2 colorectal cancer dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This will help us understand the dataset's structure and basic properties.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata fields directly (not subscripting or iterating)
metadata = dataset.metadata
print("Dataset Title:", metadata.name)
print("Dataset Description:", metadata.description)

# Display key fields
print("\nAvailable keywords:", getattr(metadata, 'keywords', None))
print("Date Published:", getattr(metadata, 'datePublished', None))
print("Dataset Version:", getattr(metadata, 'version', None))

## 2. Data Overview

Review available record sets, fields, and their `@id` values.

This section inspects how records are organized and which identifiers are available for programmatic access.

In [ ]:
# List all record sets by @id
record_sets = dataset.record_sets
print("Record Sets Available:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# To inspect fields within each record set
for rs in record_sets:
    print(f"\nFields in Record Set (@id={rs['@id']}):")
    if 'fields' in rs and isinstance(rs['fields'], list):
        for field in rs['fields']:
            print(f"  - Field @id: {field['@id']} | name: {field.get('name', 'N/A')} | dataType: {field.get('dataType', 'N/A')}")
    else:
        print("  No fields listed.")

## 3. Data Extraction

Load data from each record set into a pandas DataFrame for analysis. Use only the record set and field `@id` values from the overview above.

The following cells automatically load all available record sets, referencing them by their `@id`.

In [ ]:
# Prepare record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"\nDataFrame for record set {record_set_id}:")
    print("Columns:", dataframes[record_set_id].columns.tolist())
    print(dataframes[record_set_id].head())

# Select a record set to use for EDA
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
Use fields referred to by their `@id`.

In [ ]:
# Pick a numeric field from the first record set (by @id)
df = dataframes[main_record_set_id]

# Attempt to automatically select the first numeric-type field
record_set = next((rs for rs in dataset.record_sets if rs['@id'] == main_record_set_id), None)

numeric_field_id = None
if record_set and 'fields' in record_set:
    for field in record_set['fields']:
        if field.get('dataType') and 'Integer' in field['dataType'] or 'Float' in field['dataType']:
            numeric_field_id = field['@id']
            break

# If not found, fall back to 'Age' or similar
if not numeric_field_id:
    for col in df.columns:
        if 'age' in col.lower():
            numeric_field_id = col
            break

if numeric_field_id:
    print(f"Using numeric field (@id or column): {numeric_field_id}")
    # Try to convert to numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    threshold = 50 # example threshold for patient age
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Pick a group-by field (categorical, e.g. sex or anatomical location)
    group_field_id = None
    for field in record_set['fields']:
        if field.get('dataType')==['Text'] and 'sex' in field.get('name','').lower():
            group_field_id = field['@id']
            break

    # Default alternative
    if not group_field_id:
        for col in filtered_df.columns:
            if 'sex' in col.lower():
                group_field_id = col
                break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df)
else:
    print("No numeric field detected for this record set.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.
Below we show a histogram and, if possible, a boxplot by sex or anatomical groupings, referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=14)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we used `mlcroissant` to load and explore the FAIR^2 dataset, referencing all entities by their `@id`.

- Loaded metadata and record sets using Croissant schema.
- Extracted dataframes for each record set and examined columns using unique identifiers.
- Applied simple EDA by filtering and normalizing numeric fields such as patient age, and grouped by categorical attributes such as sex.
- Visualized distributions and groupwise comparisons for deeper analysis.

This workflow provides an extensible and reproducible way to programmatically inspect complex, FAIR datasets in clinical cancer research.